<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/calculate_Krippendorffs_alpha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import auth
auth.authenticate_user()

In [ ]:
!pip install simpledorff

import pandas as pd
import simpledorff
import numpy as np

def calculate_krippendorff_alpha_fixed():
    file_path = '/content/drive/MyDrive/world-inflation/data/reddit/production/annotation_result_after_dedup.tsv'

    print("--- 1. Loading Data ---")
    try:
        df = pd.read_csv(file_path, sep='\t')
        print(f"Loaded {len(df)} rows.")
    except Exception as e:
        print(f"CRITICAL ERROR loading file: {e}")
        return

    # --- 2. Cleaning Data ---
    print("\n--- 2. Cleaning Data ---")
    target_cols = ['RS', 'A', 'B']

    # Force convert to numeric, coercing errors to NaN
    for col in target_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Drop rows with NaN values (invalid data)
    df = df.dropna(subset=target_cols)

    # Convert to integers as alpha requires discrete categories
    df[target_cols] = df[target_cols].astype(int)

    # Assign unique unit_id based on index
    df['unit_id'] = df.index

    # --- 3. Calculating Overall Alpha ---
    print("\n--- 3. Calculating Overall Alpha ---")

    # Convert from Wide format to Long format for simpledorff
    df_long = df.melt(
        id_vars=['unit_id'],
        value_vars=['RS', 'A', 'B'],
        var_name='annotator_id',
        value_name='annotation'
    )

    try:
        # Calculate Alpha (Removed 'metric' argument as it is not supported in this function signature)
        overall_alpha = simpledorff.calculate_krippendorffs_alpha_for_df(
            df_long,
            experiment_col='unit_id',
            annotator_col='annotator_id',
            class_col='annotation'
        )
        print(f"Overall Krippendorff's Alpha: {overall_alpha:.4f}")
    except Exception as e:
        print(f"Error calculating overall alpha: {e}")

    # --- 4. Breakdown by Subreddit ---
    print("\n--- 4. Breakdown by Subreddit ---")
    print(f"{'Subreddit':<20} | {'Count':<10} | {'Alpha':<10}")
    print("-" * 45)

    subreddits = df['subreddit_id'].unique()

    for sub_id in subreddits:
        sub_df = df[df['subreddit_id'] == sub_id]

        # Skip subreddits with insufficient samples
        if len(sub_df) < 5:
            continue

        # Convert to Long Format
        sub_long = sub_df.melt(
            id_vars=['unit_id'],
            value_vars=['RS', 'A', 'B'],
            var_name='annotator_id',
            value_name='annotation'
        )

        # Check for variance (if all votes are identical, alpha cannot be calculated)
        unique_votes = sub_long['annotation'].unique()
        if len(unique_votes) < 2:
            print(f"{str(sub_id):<20} | {len(sub_df):<10} | N/A (No Variance)")
            continue

        try:
            # Calculate Alpha for specific subreddit
            sub_alpha = simpledorff.calculate_krippendorffs_alpha_for_df(
                sub_long,
                experiment_col='unit_id',
                annotator_col='annotator_id',
                class_col='annotation'
            )
            print(f"{str(sub_id):<20} | {len(sub_df):<10} | {sub_alpha:.4f}")
        except Exception as e:
            print(f"{str(sub_id):<20} | {len(sub_df):<10} | Error")

calculate_krippendorff_alpha_fixed()